### Import Dependencies

In [1]:
import os
from pathlib import Path

import instructor
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# Find the project root and load its .env file for notebook kernels started in either folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel inside the project directory.")
load_dotenv(PROJECT_ROOT / ".env", override=True)

google_api_key = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
if not google_api_key:
    raise RuntimeError("Set GOOGLE_API_KEY or GEMINI_API_KEY in the project .env.")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "Amazon-items-minilm-l6-v2"
gemini_client = genai.Client(api_key=google_api_key)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

/Users/ganesh.m/Projects/ai-engineering-bootcamp/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6678.72it/s]


### RAG Pipeline

In [2]:
client = instructor.from_provider(
    f"google/{GEMINI_MODEL}",
    mode=instructor.Mode.TOOLS,
)

In [3]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [4]:
def get_embedding(text):
    # Use the same local embedding model and Qdrant collection as the project RAG pipeline.
    return embedding_model.encode(text, normalize_embeddings=True).tolist()


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


def process_context(context):
    return "\n".join(
        f"- ID: {product_id}, rating: {rating}, description: {description}"
        for product_id, description, rating in zip(
            context["retrieved_context_ids"],
            context["retrieved_context"],
            context["retrieved_context_ratings"],
        )
    )


def build_prompt(preprocessed_context, question):
    return f"""You are a shopping assistant that answers questions about the products in stock.
Answer using only the available products below. Refer to them as available products, not as context.

Available products:
{preprocessed_context}

Question:
{question}
"""


def generate_answer(prompt):
    return client.create(
        messages=[{"role": "user", "content": prompt}],
        response_model=RAGGenerationResponse,
    )


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

In [5]:
output = rag_pipeline("Can you suggest me some earphones?")

In [6]:
output

{'data_object': RAGGenerationResponse(answer='Yes, here are some available earphones you can choose from:\n\n1. **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)** - Features a 3.5mm jack, stereo sound quality, ergonomic semi-in-ear design, built-in microphone, and remote control. Compatible with various devices like iPhones, iPads, Android smartphones, and tablets.\n2. **Volkano Ninja Kids Earphones for Boys (ID: B0BBF2VC6X)** - Wired earbuds featuring a 3.5mm stereo jack, hearing protection with a safe volume limit of 85dB, a mini carry case, and a built-in pause/play button. Available in cute designs like Ninja, Bulldog on Skateboard, Jet, and Space.\n3. **Jesebang Wireless Earbuds (ID: B09X9838WY)** - Bluetooth 5.2 wireless earphones with touch control, a built-in microphone, 13mm drivers for immersive stereo sound, a charging case with up to 40 hours of playtime, and IP7 waterproof rating. Compatible with iOS and Android devices.'),
 'answer': 'Yes, here are some available earphon

In [7]:
print(output["answer"])

Yes, here are some available earphones you can choose from:

1. **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)** - Features a 3.5mm jack, stereo sound quality, ergonomic semi-in-ear design, built-in microphone, and remote control. Compatible with various devices like iPhones, iPads, Android smartphones, and tablets.
2. **Volkano Ninja Kids Earphones for Boys (ID: B0BBF2VC6X)** - Wired earbuds featuring a 3.5mm stereo jack, hearing protection with a safe volume limit of 85dB, a mini carry case, and a built-in pause/play button. Available in cute designs like Ninja, Bulldog on Skateboard, Jet, and Space.
3. **Jesebang Wireless Earbuds (ID: B09X9838WY)** - Bluetooth 5.2 wireless earphones with touch control, a built-in microphone, 13mm drivers for immersive stereo sound, a charging case with up to 40 hours of playtime, and IP7 waterproof rating. Compatible with iOS and Android devices.


### RAG Pipeline with Grounding Context

In [8]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [16]:
def get_embedding(text):
    # Use the same local embedding model and Qdrant collection as the project RAG pipeline.
    return embedding_model.encode(text, normalize_embeddings=True).tolist()


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


def process_context(context):
    return "\n".join(
        f"- ID: {product_id}, rating: {rating}, description: {description}"
        for product_id, description, rating in zip(
            context["retrieved_context_ids"],
            context["retrieved_context"],
            context["retrieved_context_ratings"],
        )
    )


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt


def generate_answer(prompt):
    return client.create(
        messages=[{"role": "user", "content": prompt}],
        response_model=RAGGenerationResponse,
    )


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

In [17]:
output = rag_pipeline("Can you suggest me some earphones?")

In [18]:
output

{'data_object': RAGGenerationResponse(answer='Yes, here are some earphone options available in the available products:\n\n1. **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)** - Features a 3.5mm jack, Apple MFi certification, built-in microphone, volume control, and an ergonomic semi-in-ear design. Rated 4.9.\n2. **Volkano Ninja Kids Earphones (ID: B0BBF2VC6X)** - Wired earbuds with a 3.5mm stereo jack, safe volume limiting to 85dB for hearing protection, built-in mic, carry case, and cute designs. Rated 4.2.\n3. **Jesebang Wireless Earbuds (ID: B09X9838WY)** - Bluetooth 5.2 wireless earphones with touch control, built-in mic, 13mm drivers for immersive stereo sound, IP7 waterproof rating, and a charging case with up to 40 hours of playtime. Rated 4.2.', references=[RAGUsedContext(id='B0CH8DRD6K', description='2 Pack iPhone Wired Headphones with 3.5mm Jack'), RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids Earphones for Boys with Carry Case and Keyring'), RAGUsedContex

In [12]:
print(output["answer"])

Yes, here are some earphone options available in the available products:

1. **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)** - Rated 4.9. These Apple MFi Certified wired earbuds feature a 3.5mm jack, stereo sound quality, ergonomic semi-in-ear design, built-in microphone, and volume control remote. They are compatible with iPhones, iPads, iPods, Android devices, and other 3.5mm jack devices.
2. **Volkano Ninja Kids Earphones (ID: B0BBF2VC6X)** - Rated 4.2. These wired earbuds are designed specifically for kids, featuring a 3.5mm stereo jack, hearing protection with a volume limit of 85dB, a noise-isolating effect, a built-in pause/play button, and a cute carrying case and keyring. They are available in various fun designs like Ninja, Bulldog on Skateboard, Jet, and Space.
3. **Jesebang Wireless Earbuds (ID: B09X9838WY)** - Rated 4.2. These wireless Bluetooth 5.2 earbuds feature 13mm drivers for immersive stereo sound, built-in microphones, touch control, IP7 waterproof rating, and 

In [13]:
output = rag_pipeline("Can you suggest me some earphones?", top_k=10)

In [14]:
output

{'data_object': RAGGenerationResponse(answer='Here are some earphone and headphone options available among the products:\n\n- **Kids Wireless Headphones (ID: B09KQP2H7N):** Wireless and wired over-ear headphones with a 3.5mm jack, Bluetooth 5.0, volume control, foldable design, and built-in microphone (Rated 4.2).\n- **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K):** Apple MFi certified wired earbuds with a 3.5mm jack, stereo sound, ergonomic semi-in-ear design, microphone, and volume control (Rated 4.9).\n- **Sleep Headphones Bluetooth Headband (ID: B0BM657X74):** Ultra-thin wireless Bluetooth headband and sleep mask with Hi-Fi stereo speakers, built-in microphone, and 10–12 hours of playing time (Rated 4.6).\n- **Volkano Ninja Kids Earphones (ID: B0BBF2VC6X):** Wired earbuds with a 3.5mm stereo jack, safe volume limit of 85dB, carry case, and microphone (Rated 4.2).\n- **Jesebang Wireless Earbuds (ID: B09X9838WY):** Bluetooth 5.2 touch control earbuds with a built-in mic, 13mm driv

In [15]:
print(output["answer"])

Here are some earphone and headphone options available among the products:

- **Kids Wireless Headphones (ID: B09KQP2H7N):** Wireless and wired over-ear headphones with a 3.5mm jack, Bluetooth 5.0, volume control, foldable design, and built-in microphone (Rated 4.2).
- **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K):** Apple MFi certified wired earbuds with a 3.5mm jack, stereo sound, ergonomic semi-in-ear design, microphone, and volume control (Rated 4.9).
- **Sleep Headphones Bluetooth Headband (ID: B0BM657X74):** Ultra-thin wireless Bluetooth headband and sleep mask with Hi-Fi stereo speakers, built-in microphone, and 10–12 hours of playing time (Rated 4.6).
- **Volkano Ninja Kids Earphones (ID: B0BBF2VC6X):** Wired earbuds with a 3.5mm stereo jack, safe volume limit of 85dB, carry case, and microphone (Rated 4.2).
- **Jesebang Wireless Earbuds (ID: B09X9838WY):** Bluetooth 5.2 touch control earbuds with a built-in mic, 13mm drivers, IP7 waterproof rating, and a charging case prov